# 🥗 Rede Alimenta IA: MVP e Pipeline Seguro (Checkpoint 1)
**Projeto Integrador - 1º Ano de Inteligência Artificial (FIAP)**

Este notebook contém a implementação do núcleo inteligente do projeto, focado na triagem e priorização automatizada de excedentes alimentares para combate ao desperdício e à fome.

### 🛡️ Requisitos de Cibersegurança & DevSecOps aplicados:
* **Validação de Esquema:** Verificação estricta de colunas e integridade do dataset.
* **Input Validation & Sanitization:** Bloqueio de valores anómalos ou injeções de dados.
* **Tratamento Defensivo:** Proteção contra *Information Disclosure* em exceções.

## 📊 1. Carregamento Seguro e Curadoria do Dataset
Nesta etapa, carregamos o histórico consolidado de inventário e perdas (`Consolidated_Supermarket_Data.csv`). O script valida programaticamente o esquema estrutural do ficheiro para garantir que os dados não estão corrompidos antes de alimentar o modelo de Machine Learning (`RandomForestRegressor`).

In [6]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from typing import Dict, Union

class AlimentoPrioridadeSeguroModel:
    def __init__(self):
        # Inicializa o modelo de forma controlada
        self.model = RandomForestRegressor(n_estimators=100, random_state=42)
        self._is_trained = False

    def treinar_modelo(self) -> None:
        """Treina o modelo com um dataset controlado e validado."""
        try:
            dados = {
                'dias_para_vencer': [1, 5, 10, 2, 30, 0, 3, 15],
                'quantidade_kg': [50, 200, 20, 100, 500, 10, 80, 150],
                'tipo_perecivel': [1, 1, 0, 1, 0, 1, 1, 0],
                'prioridade': [95, 70, 20, 90, 10, 100, 85, 30]
            }

            df = pd.DataFrame(dados)
            X = df[['dias_para_vencer', 'quantidade_kg', 'tipo_perecivel']]
            y = df['prioridade']

            self.model.fit(X, y)
            self._is_trained = True
            print("[INFO] Modelo de IA treinado com sucesso sob diretrizes de integridade.")
        except Exception as e:
            # Segurança: Nunca expor detalhes sensíveis do erro de sistema
            raise RuntimeError("Erro crítico ao inicializar o motor de treino do modelo.")

    def prever_prioridade(self, dias_para_vencer: int, quantidade_kg: float, tipo_perecivel: int) -> Dict[str, Union[str, float]]:
        """
        Realiza a predição aplicando validação estricta de inputs (Input Sanitization)
        para mitigar potenciais injeções de dados e valores anómalos.
        """
        if not self._is_trained:
            return {"erro": "O modelo não se encontra treinado ou disponível."}

        # SEGURANÇA: Validação de Limites Lógicos (Input Validation)
        if not isinstance(dias_para_vencer, (int, float)) or dias_para_vencer < 0 or dias_para_vencer > 365:
            return {"erro": "Parâmetro inválido: 'dias_para_vencer' deve estar entre 0 e 365."}

        if not isinstance(quantidade_kg, (int, float)) or quantidade_kg <= 0 or quantidade_kg > 100000:
            return {"erro": "Parâmetro inválido: 'quantidade_kg' deve estar entre 0.1 e 100.000 kg."}

        if tipo_perecivel not in [0, 1]:
            return {"erro": "Parâmetro inválido: 'tipo_perecivel' deve ser estritamente 0 ou 1."}

        try:
            # Executa a predição de forma segura
            features = [[float(dias_para_vencer), float(quantidade_kg), int(tipo_perecivel)]]
            predicao = self.model.predict(features)

            return {
                "status": "sucesso",
                "prioridade_calculada": round(float(predicao[0]), 2)
            }
        except Exception:
            # Tratamento defensivo de excepções para evitar Information Disclosure
            return {"erro": "Ocorreu um erro interno ao processar a predição do lote."}

# Teste de Segurança Integrado
if __name__ == "__main__":
    sistema_ia = AlimentoPrioridadeSeguroModel()
    sistema_ia.treinar_modelo()

    # Teste 1: Entrada Válida
    print("--- Teste 1 (Entrada Válida) ---")
    print(sistema_ia.prever_prioridade(dias_para_vencer=2, quantidade_kg=120.5, tipo_perecivel=1))

    # Teste 2: Entrada inválida/anômala bloqueada pela validação
    print("\n--- Teste 2: Entrada Inválida/Anômala Bloqueada ---")
    print(sistema_ia.prever_prioridade(dias_para_vencer=-50, quantidade_kg=999999, tipo_perecivel=99))

[INFO] Modelo de IA treinado com sucesso sob diretrizes de integridade.
--- Teste 1 (Entrada Válida) ---
{'status': 'sucesso', 'prioridade_calculada': 89.6}

--- Teste 2: Entrada Inválida/Anômala Bloqueada ---
{'erro': "Parâmetro inválido: 'dias_para_vencer' deve estar entre 0 e 365."}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


## 🔒 2. Validação de Segurança e Teste de Robustez (Input Validation)
O teste abaixo demonstra a eficácia das medidas protetivas implementadas. Ao tentarmos inserir propositadamente valores anómalos ou fora dos limites lógicos (como quantidades negativas ou extremas), o pipeline intercepta a operação e bloqueia a entrada, garantindo a resiliência do sistema contra dados maliciosos.

### 🎯 Definição da variável-alvo

O dataset utilizado não possui uma variável específica de prioridade
para os lotes alimentares. Por isso, para este MVP, foi criada uma
pontuação de urgência derivada da taxa de perda e da quantidade
vendida.

Essa pontuação é utilizada como variável-alvo (`y`) para demonstrar
o funcionamento do modelo de Machine Learning.

In [8]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import os
from typing import Dict, Union

class PipelineAlimentoSeguro:
    def __init__(self, caminho_csv: str):
        # SEGURANÇA: Validação de Path Traversal e extensão do arquivo
        if not caminho_csv.endswith('.csv') or '..' in caminho_csv:
            raise ValueError("Erro de segurança: Caminho de arquivo inválido ou não autorizado.")

        self.caminho_csv = caminho_csv
        self.model = RandomForestRegressor(n_estimators=50, random_state=42)
        self._is_trained = False

    def carregar_e_validar_dados(self) -> pd.DataFrame:
        """Carrega o dataset aplicando validação estricta de colunas e integridade."""
        if not os.path.exists(self.caminho_csv):
            raise FileNotFoundError("O ficheiro de dados consolidado não foi encontrado.")

        try:
            # Leitura controlada do CSV
            df = pd.read_csv(self.caminho_csv)

            # SEGURANÇA: Verificação obrigatória do esquema de colunas (proteção contra dados corrompidos)
            colunas_obrigatorias = [
                'Quantity Sold (kilo)', 'Unit Selling Price (RMB/kg)',
                'Wholesale Price (RMB/kg)', 'Loss Rate (%)'
            ]

            for col in colunas_obrigatorias:
                if col not in df.columns:
                    raise KeyError(f"Coluna obrigatória em falta no dataset: {col}")

            print("[INFO] Dados carregados e validados com sucesso sob rigor de integridade.")
            return df

        except Exception as e:
            # SEGURANÇA: Evita Information Disclosure (não expõe rastros técnicos detalhados)
            raise RuntimeError("Falha crítica ao processar e validar o dataset de inventário.")

    def treinar_modelo_com_dados_reais(self) -> None:
        """Treina a IA utilizando o histórico real do supermercado de forma segura."""
        df = self.carregar_e_validar_dados()

        # Tratamento de valores nulos ou anómalos para prevenir falhas no modelo
        df_limpo = df.dropna(subset=['Quantity Sold (kilo)', 'Loss Rate (%)', 'Wholesale Price (RMB/kg)'])

        # Features baseadas no histórico real do inventário
        X = df_limpo[['Quantity Sold (kilo)', 'Wholesale Price (RMB/kg)', 'Loss Rate (%)']]

        # Alvo derivado para o MVP:
        # como o dataset não possui uma coluna de prioridade específica
        # para o projeto, criamos uma pontuação de urgência baseada
        # na taxa de perda e na quantidade vendida.
        y = (
          df_limpo['Loss Rate (%)'] * 2
          + df_limpo['Quantity Sold (kilo)'] * 0.5
        )

        self.model.fit(X, y)
        self._is_trained = True
        print("[INFO] Modelo de IA treinado com sucesso com base no histórico real.")

    def prever_prioridade_lote(self, qtd_kilo: float, preco_atacado: float, taxa_perda: float) -> Dict[str, Union[str, float]]:
        """
        Realiza predições aplicando regras estritas de Input Validation (Mitigação de Injeção).
        """
        if not self._is_trained:
            return {"erro": "O modelo não se encontra treinado."}

        # SEGURANÇA: Validação rigorosa de limites lógicos (Input Sanitization)
        if not isinstance(qtd_kilo, (int, float)) or qtd_kilo <= 0 or qtd_kilo > 10000:
            return {"erro": "Parâmetro inválido: 'qtd_kilo' fora do intervalo permitido (0 a 10.000)."}

        if not isinstance(preco_atacado, (int, float)) or preco_atacado <= 0 or preco_atacado > 1000:
            return {"erro": "Parâmetro inválido: 'preco_atacado' fora do intervalo permitido."}

        if not isinstance(taxa_perda, (int, float)) or taxa_perda < 0 or taxa_perda > 100:
            return {"erro": "Parâmetro inválido: 'taxa_perda' deve estar entre 0% e 100%."}

        try:
            # Processamento seguro com tipos explicitamente convertidos
            features = [[float(qtd_kilo), float(preco_atacado), float(taxa_perda)]]
            predicao = self.model.predict(features)

            return {
                "status": "sucesso",
                "pontuacao_urgencia": round(float(predicao[0]), 2)
            }
        except Exception:
            # Tratamento defensivo de excepções
            return {"erro": "Erro interno ao processar a predição de urgência do lote."}

# Execução do Pipeline Seguro
if __name__ == "__main__":
    # Instancia o pipeline apontando para o ficheiro consolidado
    pipeline = PipelineAlimentoSeguro("Consolidated_Supermarket_Data.csv")
    pipeline.treinar_modelo_com_dados_reais()

    # Teste 1: Lote com parâmetros válidos
    print("\n--- Teste 1: Dados Válidos ---")
    print(pipeline.prever_prioridade_lote(qtd_kilo=50.0, preco_atacado=5.50, taxa_perda=12.5))

    # Teste 2: Tentativa de inserção de dados maliciosos/anómalos (Bloqueado)
    print("\n--- Teste 2: Tentativa de Injeção de Dados Anómalos ---")
    print(pipeline.prever_prioridade_lote(qtd_kilo=-999.0, preco_atacado=0, taxa_perda=150.0))

[INFO] Dados carregados e validados com sucesso sob rigor de integridade.
[INFO] Modelo de IA treinado com sucesso com base no histórico real.

--- Teste 1: Dados Válidos ---
{'status': 'sucesso', 'pontuacao_urgencia': 25.34}

--- Teste 2: Tentativa de Injeção de Dados Anómalos ---
{'erro': "Parâmetro inválido: 'qtd_kilo' fora do intervalo permitido (0 a 10.000)."}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


### 📝 Conclusão da Evidência

* **Sucesso na Predição:** O modelo processou os dados do dataset e
  realizou uma previsão de pontuação de urgência para o lote informado.

* **Validação de Entrada:** O sistema bloqueou valores fora dos limites
  definidos, como quantidade negativa, preço igual a zero e taxa de
  perda superior a 100%.

* **Tratamento Defensivo:** Em caso de erro durante a predição, o
  sistema retorna uma mensagem genérica, evitando a exposição de
  detalhes internos da aplicação.

* **Integridade do Dataset:** Antes do treinamento, o sistema verifica
  a existência das colunas obrigatórias e remove registros com valores
  nulos nas variáveis utilizadas pelo modelo.

A implementação demonstra o funcionamento do MVP de IA e das
principais medidas de segurança aplicadas ao processamento dos dados.

# 🛡️ 3. Matriz de Ameaças e Riscos - STRIDE

A metodologia STRIDE foi utilizada para identificar ameaças de
segurança relacionadas ao sistema Rede Alimenta IA.

A análise considera as principais funcionalidades previstas para o
sistema, incluindo cadastro e processamento de doações, usuários,
organizações e utilização do modelo de IA.

In [9]:
import pandas as pd

# Matriz de ameaças STRIDE do projeto Rede Alimenta IA

stride_data = [
    {
        "Categoria": "Spoofing",
        "Ameaça": "Usuário mal-intencionado se passar por um doador ou ONG.",
        "Impacto": "Alto",
        "Mitigação": "Autenticação dos usuários e validação de identidade."
    },
    {
        "Categoria": "Tampering",
        "Ameaça": "Alteração indevida dos dados de uma doação.",
        "Impacto": "Alto",
        "Mitigação": "Validação de entradas, controle de acesso e registros de alterações."
    },
    {
        "Categoria": "Repudiation",
        "Ameaça": "Usuário negar que realizou determinada operação.",
        "Impacto": "Médio",
        "Mitigação": "Registro de operações e logs de auditoria."
    },
    {
        "Categoria": "Information Disclosure",
        "Ameaça": "Exposição de dados de doadores, ONGs ou informações internas.",
        "Impacto": "Alto",
        "Mitigação": "Controle de acesso, proteção dos dados e tratamento seguro de exceções."
    },
    {
        "Categoria": "Denial of Service",
        "Ameaça": "Envio excessivo de requisições para prejudicar a disponibilidade do sistema.",
        "Impacto": "Médio",
        "Mitigação": "Limitação de requisições, monitoramento e proteção da aplicação."
    },
    {
        "Categoria": "Elevation of Privilege",
        "Ameaça": "Usuário comum conseguir acessar funcionalidades administrativas.",
        "Impacto": "Alto",
        "Mitigação": "Controle de permissões e separação entre níveis de acesso."
    }
]

df_stride = pd.DataFrame(stride_data)

display(df_stride)

,Categoria,Ameaça,Impacto,Mitigação
0,Spoofing,Usuário mal-intencionado se passar por um doad...,Alto,Autenticação dos usuários e validação de ident...
1,Tampering,Alteração indevida dos dados de uma doação.,Alto,"Validação de entradas, controle de acesso e re..."
2,Repudiation,Usuário negar que realizou determinada operação.,Médio,Registro de operações e logs de auditoria.
3,Information Disclosure,"Exposição de dados de doadores, ONGs ou inform...",Alto,"Controle de acesso, proteção dos dados e trata..."
4,Denial of Service,Envio excessivo de requisições para prejudicar...,Médio,"Limitação de requisições, monitoramento e prot..."
5,Elevation of Privilege,Usuário comum conseguir acessar funcionalidade...,Alto,Controle de permissões e separação entre nívei...


## 🔎 Conclusão da Análise STRIDE

A análise STRIDE identificou seis categorias de ameaças relevantes
para o Rede Alimenta IA.

Os principais riscos estão relacionados à autenticação de usuários,
alteração indevida de dados, exposição de informações, disponibilidade
do sistema e controle de privilégios.

As medidas de segurança já implementadas neste MVP incluem validação
de entradas, validação da estrutura do dataset e tratamento defensivo
de exceções.

As demais medidas, como autenticação, controle de acesso, logs,
proteção contra excesso de requisições e gerenciamento de permissões,
serão aprofundadas nas próximas etapas do projeto DevSecOps.